# Authorship Attribution — Full Pipeline
**Group 7 | LIS 501**

Pipeline: EDA → Stylometric Features → Classical ML → DistilBERT → Stacked Ensemble → Demo

## 0. Imports & Setup

In [ ]:
# Core
import pandas as pd
import numpy as np
import os, re, string, warnings, pickle
warnings.filterwarnings('ignore')

# NLP
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix

# Viz
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from wordcloud import WordCloud

# Deep learning
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

# Detect device (CUDA > MPS > CPU)
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

## 1. Load Data & Passage Segmentation

In [ ]:
# Load your preprocessed sentence dataset
sentence_df = pd.read_csv('sentence_dataset.csv')
print('Loaded sentence_dataset.csv')
print(sentence_df.shape)
print(sentence_df['author'].value_counts())

In [ ]:
# ── Passage Segmentation ──────────────────────────────────────────────────────
# Group consecutive sentences (per author) into passages of N sentences.
# This gives richer stylometric signal than single sentences.

PASSAGE_SIZE = 5  # sentences per passage

def make_passages(df, passage_size=5):
    records = []
    for author, group in df.groupby('author'):
        texts = group['text'].tolist()
        for i in range(0, len(texts) - passage_size + 1, passage_size):
            chunk = texts[i:i + passage_size]
            passage = ' '.join(chunk)
            records.append({'author': author, 'text': passage})
    return pd.DataFrame(records)

passage_df = make_passages(sentence_df, PASSAGE_SIZE)
passage_df = passage_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Passages: {len(passage_df)}')
print(passage_df['author'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
# ── Class Distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Sentence-level
counts_s = sentence_df['author'].value_counts()
axes[0].barh(counts_s.index, counts_s.values, color=sns.color_palette('tab10', len(counts_s)))
axes[0].set_title('Sentences per Author', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Count')
for i, v in enumerate(counts_s.values):
    axes[0].text(v + 50, i, str(v), va='center', fontsize=9)

# Passage-level
counts_p = passage_df['author'].value_counts()
axes[1].barh(counts_p.index, counts_p.values, color=sns.color_palette('tab10', len(counts_p)))
axes[1].set_title('Passages per Author (5-sent chunks)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count')
for i, v in enumerate(counts_p.values):
    axes[1].text(v + 5, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sentence Length Distribution by Author ────────────────────────────────────
sentence_df['word_count'] = sentence_df['text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(14, 6))
authors = sentence_df['author'].unique()
palette = sns.color_palette('tab10', len(authors))

for i, author in enumerate(sorted(authors)):
    data = sentence_df[sentence_df['author'] == author]['word_count']
    data = data[data < 100]  # trim outliers for viz
    sns.kdeplot(data, label=author, color=palette[i], linewidth=2)

plt.title('Sentence Length Distribution by Author', fontsize=14, fontweight='bold')
plt.xlabel('Words per Sentence')
plt.ylabel('Density')
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig('outputs/sentence_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Word Clouds per Author ────────────────────────────────────────────────────
stop_words = set(stopwords.words('english'))

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, author in enumerate(sorted(sentence_df['author'].unique())):
    text = ' '.join(sentence_df[sentence_df['author'] == author]['text'].tolist())
    # Remove stopwords & punctuation for cloud
    words = [w.lower() for w in text.split() if w.lower() not in stop_words and w.isalpha()]
    clean = ' '.join(words)
    wc = WordCloud(width=400, height=300, background_color='white',
                   colormap='tab10', max_words=80).generate(clean)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(author, fontsize=11, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Top Words per Author (stopwords removed)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Average Sentence Length Heatmap ─────────────────────────────────────────
stats = sentence_df.groupby('author')['word_count'].agg(['mean','median','std']).round(2)
stats.columns = ['Mean Words', 'Median Words', 'Std Dev']
print(stats.sort_values('Mean Words', ascending=False))

plt.figure(figsize=(8, 5))
sns.heatmap(stats, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5)
plt.title('Sentence Length Statistics by Author', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/length_stats_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Stylometric Feature Engineering

In [ ]:
# ── Stylometric Features ──────────────────────────────────────────────────────
# These capture surface-level writing style independently of topic/content.

FUNCTION_WORDS = set([
    'the','a','an','and','but','or','nor','for','yet','so','in','on','at',
    'to','of','with','by','from','up','about','into','through','during',
    'is','are','was','were','be','been','being','have','has','had','do',
    'does','did','will','would','could','should','may','might','shall',
    'must','can','i','he','she','it','they','we','you','my','his','her',
    'its','our','your','their','this','that','these','those','which','who'
])

def extract_stylometric_features(text):
    text = str(text)
    words = text.split()
    sentences = sent_tokenize(text) if len(text) > 10 else [text]
    num_words = max(len(words), 1)
    num_chars = max(len(text), 1)
    num_sents = max(len(sentences), 1)

    # Lexical richness (Type-Token Ratio)
    unique_words = set(w.lower() for w in words if w.isalpha())
    ttr = len(unique_words) / num_words

    # Average word length
    avg_word_len = np.mean([len(w) for w in words if w.isalpha()]) if words else 0

    # Average sentence length (words)
    avg_sent_len = num_words / num_sents

    # Punctuation rates
    comma_rate    = text.count(',') / num_chars
    semicolon_rate= text.count(';') / num_chars
    exclaim_rate  = text.count('!') / num_chars
    question_rate = text.count('?') / num_chars
    dash_rate     = (text.count('—') + text.count('–') + text.count('-')) / num_chars
    quote_rate    = (text.count('"') + text.count("'")) / num_chars
    ellipsis_rate = text.count('...') / num_chars

    # Function word rate
    fw_count = sum(1 for w in words if w.lower() in FUNCTION_WORDS)
    func_word_rate = fw_count / num_words

    # Uppercase word rate (shouting / emphasis)
    upper_rate = sum(1 for w in words if w.isupper() and len(w) > 1) / num_words

    # Long word rate (>6 chars) — vocabulary richness proxy
    long_word_rate = sum(1 for w in words if len(w) > 6) / num_words

    # Short sentence rate (<8 words)
    short_sent_rate = sum(1 for s in sentences if len(s.split()) < 8) / num_sents

    return [
        ttr, avg_word_len, avg_sent_len,
        comma_rate, semicolon_rate, exclaim_rate, question_rate,
        dash_rate, quote_rate, ellipsis_rate,
        func_word_rate, upper_rate, long_word_rate, short_sent_rate
    ]

STYLO_FEATURE_NAMES = [
    'ttr', 'avg_word_len', 'avg_sent_len',
    'comma_rate', 'semicolon_rate', 'exclaim_rate', 'question_rate',
    'dash_rate', 'quote_rate', 'ellipsis_rate',
    'func_word_rate', 'upper_rate', 'long_word_rate', 'short_sent_rate'
]

print('Extracting stylometric features from passages...')
stylo_matrix = np.array([extract_stylometric_features(t) for t in passage_df['text']])
print(f'Stylometric feature matrix: {stylo_matrix.shape}')

In [ ]:
# ── Visualise Stylometric Features by Author ─────────────────────────────────
stylo_df = pd.DataFrame(stylo_matrix, columns=STYLO_FEATURE_NAMES)
stylo_df['author'] = passage_df['author'].values

# Radar / grouped bar for a few key features
key_features = ['avg_sent_len', 'ttr', 'comma_rate', 'func_word_rate', 'long_word_rate', 'exclaim_rate']
grouped = stylo_df.groupby('author')[key_features].mean()

# Normalize 0-1 for comparability
grouped_norm = (grouped - grouped.min()) / (grouped.max() - grouped.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, 6))
grouped_norm.T.plot(kind='bar', ax=ax, colormap='tab10', width=0.75)
ax.set_title('Normalised Stylometric Features by Author', fontsize=14, fontweight='bold')
ax.set_xticklabels(key_features, rotation=30, ha='right')
ax.set_ylabel('Normalised Score (0–1)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/stylometric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Classical ML Models

In [ ]:
# ── Label Encoding & Train/Test Split ────────────────────────────────────────
le = LabelEncoder()
y = le.fit_transform(passage_df['author'])
X_text = passage_df['text'].values

X_train_txt, X_test_txt, y_train, y_test, idx_train, idx_test = train_test_split(
    X_text, y, np.arange(len(y)), test_size=0.2, random_state=SEED, stratify=y
)

stylo_train = stylo_matrix[idx_train]
stylo_test  = stylo_matrix[idx_test]

print(f'Train: {len(y_train)} | Test: {len(y_test)}')
print(f'Classes: {le.classes_}')

In [ ]:
# ── TF-IDF + Character N-Gram Features ───────────────────────────────────────
# Word-level TF-IDF
tfidf_word = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2),
    max_features=50000, sublinear_tf=True,
    min_df=3, strip_accents='unicode'
)

# Character-level TF-IDF (captures punctuation / spelling style)
tfidf_char = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5),
    max_features=30000, sublinear_tf=True, min_df=3
)

X_train_word = tfidf_word.fit_transform(X_train_txt)
X_test_word  = tfidf_word.transform(X_test_txt)

X_train_char = tfidf_char.fit_transform(X_train_txt)
X_test_char  = tfidf_char.transform(X_test_txt)

# Scale stylometric features then stack with TF-IDF
scaler = MinMaxScaler()
stylo_train_scaled = scaler.fit_transform(stylo_train)
stylo_test_scaled  = scaler.transform(stylo_test)

X_train_full = hstack([X_train_word, X_train_char, csr_matrix(stylo_train_scaled)])
X_test_full  = hstack([X_test_word,  X_test_char,  csr_matrix(stylo_test_scaled)])

print(f'Full feature matrix (train): {X_train_full.shape}')

In [ ]:
# ── Train Classical Models ────────────────────────────────────────────────────
def evaluate_model(name, model, X_train, X_test, y_train, y_test, le):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='macro')
    print(f'\n── {name} ──')
    print(f'Accuracy : {acc:.4f}')
    print(f'Macro F1 : {f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    return model, y_pred, acc, f1

# Logistic Regression
lr_model = LogisticRegression(C=5, max_iter=1000, solver='saga', n_jobs=-1, random_state=SEED)
lr_model, lr_preds, lr_acc, lr_f1 = evaluate_model(
    'Logistic Regression', lr_model, X_train_full, X_test_full, y_train, y_test, le
)

# Complement Naive Bayes (better than MultinomialNB for text imbalance)
nb_model = ComplementNB(alpha=0.1)
nb_model, nb_preds, nb_acc, nb_f1 = evaluate_model(
    'Complement Naive Bayes', nb_model, X_train_full, X_test_full, y_train, y_test, le
)

# Linear SVM (strongest classical baseline for text)
svm_model = LinearSVC(C=1.0, max_iter=2000, random_state=SEED)
svm_model, svm_preds, svm_acc, svm_f1 = evaluate_model(
    'Linear SVM', svm_model, X_train_full, X_test_full, y_train, y_test, le
)

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
model_results = [
    ('Logistic Regression', lr_preds),
    ('Naive Bayes', nb_preds),
    ('Linear SVM', svm_preds)
]
short_labels = [a.replace('William', 'W.').replace('FScott', 'F.S.') for a in le.classes_]

for ax, (name, preds) in zip(axes, model_results):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('outputs/classical_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Top TF-IDF Features per Author (LR coefficients) ─────────────────────────
feature_names = (
    tfidf_word.get_feature_names_out().tolist() +
    tfidf_char.get_feature_names_out().tolist() +
    STYLO_FEATURE_NAMES
)

fig, axes = plt.subplots(2, 5, figsize=(22, 10))
axes = axes.flatten()

for i, (author, coef) in enumerate(zip(le.classes_, lr_model.coef_)):
    top_idx = np.argsort(coef)[-15:][::-1]
    top_feats = [feature_names[j] for j in top_idx]
    top_vals  = coef[top_idx]
    axes[i].barh(top_feats[::-1], top_vals[::-1],
                 color=sns.color_palette('tab10')[i])
    axes[i].set_title(author, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('LR Coefficient')

plt.suptitle('Top Discriminative Features per Author (Logistic Regression)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/top_features_per_author.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save classical models
pickle.dump(lr_model,  open('models/lr_model.pkl',  'wb'))
pickle.dump(nb_model,  open('models/nb_model.pkl',  'wb'))
pickle.dump(svm_model, open('models/svm_model.pkl', 'wb'))
pickle.dump(tfidf_word, open('models/tfidf_word.pkl', 'wb'))
pickle.dump(tfidf_char, open('models/tfidf_char.pkl', 'wb'))
pickle.dump(scaler, open('models/scaler.pkl', 'wb'))
pickle.dump(le, open('models/label_encoder.pkl', 'wb'))
print('Classical models saved.')

## 5. DistilBERT Fine-Tuning

In [ ]:
# ── Dataset Class ─────────────────────────────────────────────────────────────
BERT_MAX_LEN = 256  # passages fit comfortably; longer = slower
BERT_BATCH   = 16
BERT_EPOCHS  = 4

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class AuthorDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = AuthorDataset(X_train_txt, y_train, tokenizer, BERT_MAX_LEN)
test_dataset  = AuthorDataset(X_test_txt,  y_test,  tokenizer, BERT_MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BERT_BATCH, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BERT_BATCH, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# ── Model Setup ───────────────────────────────────────────────────────────────
NUM_CLASSES = len(le.classes_)

bert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=NUM_CLASSES
).to(DEVICE)

optimizer = AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)

total_steps    = len(train_loader) * BERT_EPOCHS
warmup_steps   = total_steps // 10
scheduler      = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f'Model parameters: {sum(p.numel() for p in bert_model.parameters()):,}')
print(f'Training steps: {total_steps} | Warmup: {warmup_steps}')

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            probs  = torch.softmax(outputs.logits, dim=-1)
            preds  = probs.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels), np.array(all_probs)

# ── Run Training ─────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc = 0

for epoch in range(BERT_EPOCHS):
    tr_loss, tr_acc = train_epoch(bert_model, train_loader, optimizer, scheduler, DEVICE)
    vl_loss, vl_acc, bert_preds, bert_labels, bert_probs = eval_epoch(bert_model, test_loader, DEVICE)

    train_losses.append(tr_loss); val_losses.append(vl_loss)
    train_accs.append(tr_acc);   val_accs.append(vl_acc)

    print(f'Epoch {epoch+1}/{BERT_EPOCHS} | '
          f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | '
          f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}')

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(bert_model.state_dict(), 'models/best_bert.pt')
        np.save('models/bert_test_probs.npy', bert_probs)
        print(f'  ✓ Best model saved (val acc = {best_val_acc:.4f})')

In [ ]:
# ── Training Curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, BERT_EPOCHS + 1)

axes[0].plot(epochs_range, train_losses, 'o-', label='Train', color='steelblue')
axes[0].plot(epochs_range, val_losses,   'o-', label='Val',   color='tomato')
axes[0].set_title('DistilBERT Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(epochs_range, train_accs, 'o-', label='Train', color='steelblue')
axes[1].plot(epochs_range, val_accs,   'o-', label='Val',   color='tomato')
axes[1].set_title('DistilBERT Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/bert_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

bert_f1 = f1_score(bert_labels, bert_preds, average='macro')
bert_acc = accuracy_score(bert_labels, bert_preds)
print(f'\nDistilBERT — Accuracy: {bert_acc:.4f} | Macro F1: {bert_f1:.4f}')
print(classification_report(bert_labels, bert_preds, target_names=le.classes_))

In [ ]:
# ── BERT Confusion Matrix ─────────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
cm = confusion_matrix(bert_labels, bert_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('DistilBERT Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('outputs/bert_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Stacked Hybrid Ensemble

In [ ]:
# ── Generate Probability Outputs from Each Base Learner ───────────────────────
# The stacked ensemble takes probability vectors from:
#   (1) Stylometric Logistic Regression
#   (2) TF-IDF SVM (Platt scaling via CalibratedClassifierCV)
#   (3) DistilBERT
# ...and feeds them into a meta Logistic Regression (the 'stack').

from sklearn.calibration import CalibratedClassifierCV

# (1) Stylometric-only LR (captures explicit writing style)
stylo_lr = LogisticRegression(C=1.0, max_iter=1000, solver='saga', random_state=SEED)
stylo_lr.fit(stylo_train_scaled, y_train)
stylo_probs_test = stylo_lr.predict_proba(stylo_test_scaled)

# (2) SVM with calibration for probabilities
cal_svm = CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=SEED), cv=3)
cal_svm.fit(X_train_full, y_train)
svm_probs_test = cal_svm.predict_proba(X_test_full)

# (3) Load best BERT probabilities
bert_probs_test = np.load('models/bert_test_probs.npy')

print(f'Stylo probs shape : {stylo_probs_test.shape}')
print(f'SVM probs shape   : {svm_probs_test.shape}')
print(f'BERT probs shape  : {bert_probs_test.shape}')

In [ ]:
# ── Build Stacked Meta-Features & Train Meta-Learner ─────────────────────────
# Concatenate all probability vectors → input to meta LR
X_meta_test = np.hstack([stylo_probs_test, svm_probs_test, bert_probs_test])

# For training the meta-learner we need train-set probabilities.
# We get these via cross-val to avoid leakage.
from sklearn.model_selection import cross_val_predict

stylo_probs_train = cross_val_predict(
    LogisticRegression(C=1.0, max_iter=1000, solver='saga', random_state=SEED),
    stylo_train_scaled, y_train, cv=5, method='predict_proba'
)
svm_probs_train = cross_val_predict(
    CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=SEED), cv=3),
    X_train_full, y_train, cv=5, method='predict_proba'
)

# For BERT train probs: run inference on training set
bert_model.load_state_dict(torch.load('models/best_bert.pt', map_location=DEVICE))
train_dataset_inf = AuthorDataset(X_train_txt, y_train, tokenizer, BERT_MAX_LEN)
train_loader_inf  = DataLoader(train_dataset_inf, batch_size=BERT_BATCH, shuffle=False)
_, _, _, _, bert_probs_train = eval_epoch(bert_model, train_loader_inf, DEVICE)

X_meta_train = np.hstack([stylo_probs_train, svm_probs_train, bert_probs_train])

# Train meta Logistic Regression
meta_lr = LogisticRegression(C=1.0, max_iter=1000, solver='saga', random_state=SEED)
meta_lr.fit(X_meta_train, y_train)

ensemble_preds = meta_lr.predict(X_meta_test)
ensemble_probs = meta_lr.predict_proba(X_meta_test)

ens_acc = accuracy_score(y_test, ensemble_preds)
ens_f1  = f1_score(y_test, ensemble_preds, average='macro')
print(f'\nStacked Ensemble — Accuracy: {ens_acc:.4f} | Macro F1: {ens_f1:.4f}')
print(classification_report(y_test, ensemble_preds, target_names=le.classes_))

pickle.dump(meta_lr, open('models/meta_lr.pkl', 'wb'))
pickle.dump(stylo_lr, open('models/stylo_lr.pkl', 'wb'))
pickle.dump(cal_svm, open('models/cal_svm.pkl', 'wb'))

In [ ]:
# ── Ensemble Confusion Matrix ─────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
cm_ens = confusion_matrix(y_test, ensemble_preds)
sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Stacked Ensemble Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('outputs/ensemble_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Final Model Comparison

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'Logistic Regression', 'Linear SVM', 'DistilBERT', 'Stacked Ensemble'],
    'Accuracy': [nb_acc, lr_acc, svm_acc, bert_acc, ens_acc],
    'Macro F1': [nb_f1,  lr_f1,  svm_f1,  bert_f1,  ens_f1]
}).sort_values('Macro F1', ascending=False)

print(results.to_string(index=False))
results.to_csv('outputs/model_comparison.csv', index=False)

In [ ]:
# ── Comparison Bar Chart ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results))
width = 0.35
colors_acc = ['#4C72B0'] * len(results)
colors_f1  = ['#DD8452'] * len(results)
# highlight best
best_idx = results['Macro F1'].idxmax()

bars1 = ax.bar(x - width/2, results['Accuracy'], width, label='Accuracy', color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x + width/2, results['Macro F1'], width, label='Macro F1', color='#DD8452', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(results['Model'], rotation=20, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy & Macro F1', fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(0.9, color='red', linestyle='--', alpha=0.4, label='0.90 threshold')
plt.tight_layout()
plt.savefig('outputs/model_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-Author F1 Comparison ──────────────────────────────────────────────────
from sklearn.metrics import f1_score as f1

per_author = pd.DataFrame(index=le.classes_)
for name, preds in [('NB', nb_preds), ('LR', lr_preds), ('SVM', svm_preds),
                    ('BERT', bert_preds), ('Ensemble', ensemble_preds)]:
    scores = f1(y_test, preds, average=None)
    per_author[name] = scores

plt.figure(figsize=(14, 6))
per_author.plot(kind='bar', figsize=(14, 6), colormap='tab10', edgecolor='white')
plt.title('Per-Author F1 Score by Model', fontsize=14, fontweight='bold')
plt.xlabel('Author')
plt.ylabel('F1 Score')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Model', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig('outputs/per_author_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Research Questions — Answers from Results

In [ ]:
# ── RQ1: Can stylometric features alone distinguish authors? ──────────────────
stylo_only_preds = stylo_lr.predict(stylo_test_scaled)
stylo_acc = accuracy_score(y_test, stylo_only_preds)
stylo_f1  = f1_score(y_test, stylo_only_preds, average='macro')
print('RQ1 — Stylometric features only:')
print(f'  Accuracy: {stylo_acc:.4f} | Macro F1: {stylo_f1:.4f}')
print()

# ── RQ2: Does BERT outperform classical ML? ───────────────────────────────────
print('RQ2 — BERT vs best classical:')
print(f'  Best classical (SVM) F1 : {svm_f1:.4f}')
print(f'  DistilBERT F1           : {bert_f1:.4f}')
print(f'  Delta                   : {bert_f1 - svm_f1:+.4f}')
print()

# ── RQ3: Does the stacked ensemble beat individual models? ────────────────────
print('RQ3 — Stacked ensemble vs individual:')
print(f'  Best single model F1 : {max(svm_f1, bert_f1):.4f}')
print(f'  Ensemble F1          : {ens_f1:.4f}')
print(f'  Delta                : {ens_f1 - max(svm_f1, bert_f1):+.4f}')